In [8]:
import pandas as pd
import re

In [9]:
df_b1 = pd.read_csv("../data/processed/bioshock_1_clean.csv", parse_dates=["review_date"])
df_b2 = pd.read_csv("../data/processed/bioshock_2_clean.csv", parse_dates=["review_date"])
df_inf = pd.read_csv("../data/processed/bioshock_infinite_clean.csv", parse_dates=["review_date"])

In [10]:
THEMES = {
    "narrative":["twist", "would you kindly", "betrayal", "obedience", "protector", "flashback", "redemption", "timeline", "multiverse", "realities", "constants", "genes", "genetic engineering", "gene altering", "autonomy", "agency", "conditioned", "kidnapping", "storytelling", "narrative", "writing", "dialogue", "story", "plot", "ending"],
    "setting" :["rapture", "underwater", "city", "art deco", "retro", "theatre", "columbia", "floating", "sky", "americana", "steampunk", "utopia", "environment", "architecture", "dystopian", "ocean", "ruins", "aesthetic", "lighthouse", "pavilion", "arcadia", "prometheus", "amusements"],
    "atmosphere" :["horror", "eerie", "scary", "tense", "immersive", "uncomfortable", "disturbing", "mystery", "dread", "claustrophobic", "twisted", "unsettling"],
    "characters" :["jack", "ryan", "atlas", "fontaine", "tenenbaum", "cohen", "little sister", "daddies", "daddy", "splicer", "delta", "eleanor", "lamb", "sofia", "sinclair", "grace", "big sister", "poole", "booker", "elizabeth", "comstock", "songbird", "twins", "daisy", "turrets", "mosquito", "zeppelin", "barrage", "handyman", "fireman", "zealot", "motorized patriot", "siren", "boy of silence", "lutece", "suchong"],
    "combat" :["plasmids", "combat", "movement", "quantum tear", "weapons", "gunplay", "shooting", "vigors", "guns", "wrench", "research camera", "drill", "rivet", "hack", "sky-hook", "adam", "eve", "tonics", "difficulty"],
    "ideology" :["capitalism", "greed", "individualism", "free will", "wealth disparity", "exploitation", "slavery", "selfishness", "political", "religious", "religion", "oppression", "collectivism", "utilitarianism", "exceptionalism", "communism", "police state", "ayn rand", "authoritarian", "right wing", "morality", "objectivism", "racism", "nationalism"],
    "visual_audio" :["lighting", "audio", "graphics", "music", "voice acting", "sound design", "style", "soundtrack"],
    "technical" :["crash", "stutter", "performance", "port", "patch", "fov", "mouse", "bugs", "optimization", "fps", "controls", "resolution", "freezing", "freeze", "glitching", "glitch", "glitches", "crashes", "crashed", "crashing", "laggy"],
    "pacing" :["repetitive", "short", "long", "boring", "tedious", "replayability", "backtrack", "drags", "filler", "padding"],
}

In [12]:
def detect_themes(text, themes=THEMES):
    text = str(text).lower()
    return {name: any(re.search(rf"\b{re.escape(kw)}\b", text) for kw in keywords)
            for name, keywords in themes.items()}

In [13]:
detect_themes("The story twist in Rapture blew my mind, but it crashes constantly") # example review

{'narrative': True,
 'setting': True,
 'atmosphere': False,
 'characters': False,
 'combat': False,
 'ideology': False,
 'visual_audio': False,
 'technical': True,
 'pacing': False}

In [14]:
for frame in (df_b1, df_b2, df_inf):
    theme_flags = frame["review"].apply(detect_themes).apply(pd.Series)
    frame[theme_flags.columns] = theme_flags

In [15]:
theme_cols = list(THEMES.keys())

salience = pd.DataFrame({
    "BioShock 1": df_b1[theme_cols].mean(),
    "BioShock 2": df_b2[theme_cols].mean(),
    "Infinite": df_inf[theme_cols].mean(),
})
(salience * 100).round(1)

,BioShock 1,BioShock 2,Infinite
narrative,24.6,23.5,40.4
setting,8.6,8.3,9.3
atmosphere,4.9,2.6,3.0
characters,6.0,12.5,9.3
combat,10.0,13.2,13.2
ideology,1.3,1.0,1.9
visual_audio,11.1,8.2,12.1
technical,25.0,40.3,7.7
pacing,4.9,5.3,7.5
